# SH01 local AI compose on Kaggle

This notebook assumes the repository contains the AI composition patch. It starts the existing SH01 server, enables the high-resource Kaggle worker, submits a local source path, polls the durable job, and downloads the result.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time, requests

REPO = Path.cwd()
INPUT_DIR = Path('/kaggle/input/your-dataset')
ASSET_DIR = Path('/kaggle/input/asset-library')
SFX_DIR = Path('/kaggle/input/sfx-library')
SOURCE = INPUT_DIR / 'main.mp4'
OUTPUT_DIR = Path('/kaggle/working/ai-compose-output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE.exists(), SOURCE
print('repo:', REPO)
print('source:', SOURCE, SOURCE.stat().st_size)

In [ ]:
# Install dependencies and apply the additive route patch.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, 'install_ai_compose.py'], check=True)
# Optional in-process GPU director. Leave unset when using AI_COMPOSE_LLM_URL instead.
# os.environ['AI_COMPOSE_HF_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'bitsandbytes'], check=True)

In [ ]:
# Set secrets through Kaggle Secrets or the notebook environment; do not print them.
os.environ.update({
    'MEDIA_LOCAL_SOURCE_DIR': str(INPUT_DIR),
    'AI_COMPOSE_WORK_DIR': '/kaggle/working/ai-compose-jobs',
    'AI_COMPOSE_MAX_SOURCE_BYTES': str(16 * 1024**3),
    'MEDIA_ALLOW_UNLIMITED_RESOURCES': 'true',
    'MEDIA_QUEUE_CONSUMER_ENABLED': os.getenv('MEDIA_QUEUE_CONSUMER_ENABLED', 'true'),
    'MEDIA_JOB_STORE_MODE': os.getenv('MEDIA_JOB_STORE_MODE', 'remote'),
    'MEDIA_CLIP_API_KEY': os.environ['MEDIA_CLIP_API_KEY'],
    'MEDIA_JOB_STORE_API_KEY': os.environ.get('MEDIA_JOB_STORE_API_KEY', ''),
    'CONTROL_PLANE_BASE_URL': os.environ.get('CONTROL_PLANE_BASE_URL', 'https://infra-control.vivojaymail.workers.dev'),
    'AI_COMPOSE_LOCAL_FALLBACK': 'true',
    'PEXELS_API_KEY': os.environ.get('PEXELS_API_KEY', ''),
    'PIXABAY_API_KEY': os.environ.get('PIXABAY_API_KEY', ''),
    'UNSPLASH_ACCESS_KEY': os.environ.get('UNSPLASH_ACCESS_KEY', ''),
})
if ASSET_DIR.exists(): os.environ['AI_COMPOSE_ASSET_LIBRARY'] = str(ASSET_DIR)
if SFX_DIR.exists(): os.environ['AI_COMPOSE_SFX_LIBRARY'] = str(SFX_DIR)
# Run one Uvicorn worker so the existing heavy-media lock remains authoritative.
server_log = open('/kaggle/working/sh01_ai_compose_server.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '5000', '--workers', '1'], stdout=server_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:5000/openapi.json', timeout=3).ok: break
    except requests.RequestException: pass
    time.sleep(2)
print('server pid:', server.pid)

In [ ]:
BASE = 'http://127.0.0.1:5000'
headers = {'X-API-KEY': os.environ['MEDIA_CLIP_API_KEY']}
payload = {
    'unlimited_resources': True,
    'source_path': str(SOURCE),
    'brief': 'Create a highly informative, fast-paced short. Use a compelling opening, relevant supplementary visuals, clear captions, restrained adaptive whooshes, and a clean outro. Never add unrelated stock footage.',
    'preset': 'viral_short',
    'target_seconds': 60,
    'output_count': 2,
    'fetch_assets': True,
    'asset_sources': {'local_library': True, 'pexels': bool(os.environ.get('PEXELS_API_KEY')), 'pixabay': bool(os.environ.get('PIXABAY_API_KEY')), 'unsplash': False},
    'local_asset_library': str(ASSET_DIR) if ASSET_DIR.exists() else None,
    'local_sfx_library': str(SFX_DIR) if SFX_DIR.exists() else None,
    'sfx': {'mode': 'adaptive'},
}
response = requests.post(BASE + '/media/ai-compose', headers={**headers, 'Content-Type': 'application/json'}, json=payload, timeout=120)
response.raise_for_status()
job = response.json()
job

In [ ]:
status_url = BASE + f"/media/ai-compose-jobs/{job['job_id']}"
while True:
    status = requests.get(status_url, headers=headers, timeout=60).json()
    print(status.get('stage'), status.get('progress'), status.get('error'))
    if status.get('status') in {'complete', 'failed'}: break
    time.sleep(10)
if status.get('status') != 'complete': raise RuntimeError(status)
for index, output in enumerate(status.get('outputs', []), 1):
    if output.get('video_url'):
        data = requests.get(output['video_url'], timeout=900).content
    else:
        filename = f'variant_{index:02d}.mp4'
        data = requests.get(BASE + f'/media/ai-compose-files/{job["job_id"]}/{filename}', headers=headers, timeout=900).content
    target = OUTPUT_DIR / f'variant_{index:02d}.mp4'
    target.write_bytes(data)
    print(target, target.stat().st_size)
print(json.dumps(status.get('plan', {}), indent=2)[:4000])